In [2]:
# Cardialyse ECG Batch Processing Pipeline
# Full workflow: ECG Folder → Signal → R-Peaks → HRV Features

# Install required packages (run once)
!pip install numpy pandas matplotlib opencv-python scikit-image scipy neurokit2

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.morphology import skeletonize
from skimage.filters import threshold_otsu
from scipy.signal import butter, filtfilt
import pandas as pd
import neurokit2 as nk
from glob import glob

## Folder setup
img_folder = '../major project/ECG Images of Myocardial Infarction Patients (240x12=2880)'               # Update to your ECG folder path
output_folder = '../output/features/'
os.makedirs(output_folder, exist_ok=True)
img_paths = glob(os.path.join(img_folder, '*.jpg'))

## Functions (same as original)
def preprocess_ecg(img, clahe=True):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if clahe:
        clahe_obj = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        gray = clahe_obj.apply(gray)
    gray = cv2.medianBlur(gray, 3)
    return gray

def extract_waveform(gray_img):
    thresh = threshold_otsu(gray_img)
    bw = gray_img < thresh
    skeleton = skeletonize(bw)
    xs, ys = [], []
    h, w = skeleton.shape
    for x in range(w):
        rows = np.where(skeleton[:, x])[0]
        if len(rows) == 0:
            continue
        y_med = int(np.median(rows))
        xs.append(x)
        ys.append(y_med)
    return np.array(xs), np.array(ys), skeleton

def pixels_to_signal(xs, ys, pixels_per_mm, paper_speed=25.0, desired_fs=500):
    fs_est = pixels_per_mm * paper_speed
    dt = 1.0 / fs_est
    t_pixels = xs * dt
    baseline = np.median(ys)
    mV_per_pixel = 1.0 / (10.0 * pixels_per_mm)
    amps_mV = (baseline - ys) * mV_per_pixel
    t_uniform = np.arange(t_pixels.min(), t_pixels.max(), 1.0/desired_fs)
    amps_uniform = np.interp(t_uniform, t_pixels, amps_mV)
    return t_uniform, amps_uniform, fs_est

def butter_bandpass(lowcut, highcut, fs, order=3):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return b, a

def filter_ecg(signal, fs, lowcut=0.5, highcut=40.0):
    b, a = butter_bandpass(lowcut, highcut, fs)
    filtered = filtfilt(b, a, signal)
    return filtered

## Batch processing loop
pixels_per_mm = 10
paper_speed = 25.0
desired_fs = 500
all_features = []
for img_path in img_paths:
    # your existing code
    hrv_features = nk.hrv(info, sampling_rate=fs, show=False)
    hrv_features['image'] = os.path.basename(img_path)
    hrv_features['mean_hr'] = np.mean(heart_rate) if len(heart_rate) > 0 else np.nan
    if not hrv_features.empty:
        all_features.append(hrv_features)
    else:
        print(f"Warning: {img_path} produced empty features, skipping.")

if all_features:
    features_df = pd.concat(all_features, axis=0, ignore_index=True)
    features_df.to_csv(os.path.join(output_folder, 'ecg_features_batch.csv'), index=False)
    print('Batch features saved:', os.path.join(output_folder, 'ecg_features_batch.csv'))
else:
    print("No valid features to concatenate. Check input images and processing steps.")







No valid features to concatenate. Check input images and processing steps.
